In [1]:
import os
import glob
import sys

import ROOT
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tabulate import tabulate
from yaml import safe_load, YAMLError
import pickle
import config as cfg 
from efficiency_tools import efficiency_finder
import basic_functions
from df_makers import data_to_pickle_function
from apply_bdt_and_pickle_df import load_bdt_and_apply

ROOT.DisableImplicitMT() 



Welcome to JupyROOT 6.28/10


Warning in <ROOT_TImplicitMT_DisableImplicitMT>: Implicit multi-threading is already disabled


In [2]:
sq_BDT_cut = 0.99965
samples = ["p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu"]

#path to data and outputs
inputpath    = basic_functions.check_inputpath(f"/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/{samples[0]}/chunk_2_with_evtid.root")
yamlpath     = basic_functions.check_inputpath(cfg.fccana_opts['yamlPath'])
#samples = cfg.sample_allocations["Bu2lnu_background"]
#runmode = "Bu2lnu_background_no_lepton_veto"

# print statements to check loading things expect
print(f"----> INFO: Loading files from")
print(f"{15*' '}{inputpath}")



----> INFO: Loading files from
               /r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu/chunk_2_with_evtid.root


In [3]:
#Get list of vars to save
bdtvars_list_optimised = cfg.optimised_bdt_lh_opts["mvaBranchList"]
responsevars = ["EVT_hemisEmin_Emiss"] 

bdtvars      = list(set(basic_functions.vars_fromyaml(yamlpath, bdtvars_list_optimised)))
flavtag_vars = list(basic_functions.vars_fromyaml(yamlpath, "flavour-tag-vars"))
truth_vars = list(basic_functions.vars_fromyaml(yamlpath,"MCtruth-vars"))

vars_to_save = list(set(bdtvars+truth_vars+["EVT_hemisEmin_nLept"]+ responsevars))

vars_to_save = vars_to_save + ["evt_id"] + ["chunk"]

In [4]:
# list of inputs to say which BDT to use
BDT_params = {"config_bdtopts": cfg.optimised_bdt_lh_opts,
                   "training_round": "baseline-plus-hps",
                   "hps_dict_name":"baseline-plus-hps",
                   "features_list_name": "bdtlh-vars-v1",
                   "bdt_label": "_lh"}




In [5]:
cut = "EVT_hemisEmin_nLept == 0"

#apply BDT and cut

eff_bdtlh_cut = {}
N_before_bdtlh_cut = {}

# now collect relevant events into a dataframe
decay = samples[0]
print(f'Starting processing decay: {decay}')
N_pre=0
N_post=0

nchunks = 1
chunked_populated_files = [inputpath]


for n in range(nchunks):
    print(f'---> Starting processing chunk {n} of {nchunks}')
    files =chunked_populated_files[n]
    Rdf = ROOT.RDataFrame("events", files)
    if cut:
        Rdf = Rdf.Filter(cut)
    Rdf_np = Rdf.AsNumpy(columns= vars_to_save)
    sub_df = pd.DataFrame(Rdf_np)
    sub_df["decay"] = decay

    # want to make sure that integer types are actually set as integers - currenlty stored as float
    #if changed branches significantly might be worth checking the list is still right, with current branches expected integers in yaml
    integer_branches = [s for s in vars_to_save if '_n' in s and '_norm' not in s]
    for integer_branch in integer_branches:
        sub_df[integer_branch] = sub_df[integer_branch].astype(np.int32)

    #####################
    # apply BDT and cut #
    #####################
    
    model, bdtname, dataframe_chunk = load_bdt_and_apply(sub_df, 
                        config_bdtopts = BDT_params["config_bdtopts"],
                        training_round = BDT_params["training_round"],
                        hps_dict_name = BDT_params["hps_dict_name"],
                        features_list_name = BDT_params["features_list_name"],
                        bdt_label = BDT_params["bdt_label"])
        

    N_pre += len(dataframe_chunk)
    cut_df_chunk = dataframe_chunk[((1-dataframe_chunk['bdt_score_1'])>sq_BDT_cut)&((1-dataframe_chunk['bdt_score_0'])>sq_BDT_cut)]
    N_post += len(cut_df_chunk)


    N_before_bdtlh_cut[decay] = N_pre 
    eff_bdtlh_cut[decay] = N_post/N_pre 


cut_df_chunk



Starting processing decay: p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu
---> Starting processing chunk 0 of 1
Loading BDT model...
BDT model loaded successfully.


,EVT_hemisEmin_eCharged,PV_Rec_vtx_m,Rec_vtx_n,ln_Rec_vtx_d2PV_max_hemisEmax,MCq2_e,Rec_true_M1ofM2,Rec_true_M2ofM1,ln_Rec_vtx_d2PV_min_hemisEmin,MCZ_py,Rec_true_orivtx_y,...,EVT_hemisEmin_nNeutral,EVT_Thrust_mag,MC_D2,Rec_true_q,evt_id,chunk,decay,bdt_score_0,bdt_score_1,bdt_score_2
4,3.738137,5.842944,3,2.088527,45.594002,"[0, 0, 5, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, ...","[21, 0, 0, 0, 0, 0, 0, 0, 0, 21, 0, 21, 0, 0, ...",10.0,0.0,"[-1.2022553164570127e-05, -4.170631408691406, ...",...,5,0.961405,"[-999, -999, -999, -999, 8, -999, -999, 10, -9...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1...",86,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000026,0.000063,0.999911
75,4.042670,13.171070,3,1.149852,45.593998,"[0, 0, 21, 21, 0, 21, 0, 0, 0, 0, 0, 0, 21, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",10.0,0.0,"[-1.108461618423462, -1.0909417867660522, -1.7...",...,6,0.904353,"[-999, -999, -999, -999, 8, -999, -999, -999, ...","[-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1...",1338,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000277,0.000319,0.999404
111,2.444443,1.778099,3,2.009313,40.408772,"[0, 0, 0, 0, 0, 0, 0, 0, 21, 21, 0, 0, 0, 0, 0...","[21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",10.0,0.0,"[-3.741881300811656e-05, -4.992905616760254, -...",...,3,0.956558,"[-999, -999, -999, -999, -999, -999, 3, 12, -9...","[1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1...",2474,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000008,0.000012,0.999979
192,6.425947,13.147729,4,4.321993,45.593998,"[0, 0, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[21, 0, 0, 0, 0, 0, 0, 0, 0, 21, 0, 21, 0, 0, ...",10.0,0.0,"[-1.6008101738407277e-05, -56.86274719238281, ...",...,12,0.926314,"[-999, -999, -999, -999, 8, -999, -999, 10, -9...","[1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0, ...",4334,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000219,0.000237,0.999544
201,4.767282,11.835362,2,1.318806,45.584801,"[0, 0, 0, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[21, 21, 21, 0, 0, 0, 0, 21, 21, 21, 21, 21, 0...",10.0,0.0,"[2.0955500076524913e-05, 2.0955500076524913e-0...",...,4,0.911084,"[-999, -999, -999, -999, -999, -999, 3, 12, -9...","[-1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1...",4581,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000077,0.000062,0.999861
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3869,9.933797,9.136457,3,1.038196,45.593998,"[0, 0, 0, 0, 21, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[-3, -3, -3, -3, 0, 0, -3, 21, 0, 0, 0, 0, 0, ...",10.0,0.0,"[4.207347592455335e-05, 4.2053969082189724e-05...",...,8,0.866720,"[-999, -999, -999, -999, 8, -999, -999, -999, ...","[1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1....",91130,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000058,0.000219,0.999723
3879,1.358687,3.715876,3,0.279492,45.593998,"[21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 21, 0, 0, 0, 21, 21, 0, 21, -3, -3, -3, 0,...",10.0,0.0,"[3.336581721669063e-05, 3.336581721669063e-05,...",...,3,0.894477,"[-999, -999, -999, -999, 8, -999, -999, -999, ...","[1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1...",91261,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000194,0.000089,0.999716
3886,3.283550,4.350756,3,1.537866,42.593513,"[0, 21, 21, 21, 0, 0, 0, 0, -5, 0, 0, 0, 0, 0,...","[21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 21, 0...",10.0,0.0,"[2.11965980270179e-05, 2.11965980270179e-05, 2...",...,5,0.955100,"[-999, -999, -999, -999, -999, 2, -999, 12, -9...","[-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1....",91521,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000009,0.000074,0.999917
3891,3.646395,7.085983,4,5.736484,45.592312,"[21, 0, 21, 0, 0, 0, 0, 0, 0, 0, 0, 21, 0, 21,...","[0, 21, 0, 21, 0, 0, 0, 0, 0, 0, 21, 0, 0, 0, ...",10.0,0.0,"[-1.0814820598170627e-05, -1.0814820598170627e...",...,8,0.951769,"[-999, -999, -999, -999, -999, 2, -999, 12, -9...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, ...",91565,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000088,0.000118,0.

In [6]:
print(list(cut_df_chunk["evt_id"]))

[86, 1338, 2474, 4334, 4581, 4984, 5085, 5121, 5250, 5689, 6483, 6554, 6920, 7300, 7626, 7714, 8629, 9084, 9244, 9383, 9661, 9976, 10164, 11275, 12092, 12494, 12733, 13669, 14304, 15178, 15474, 16551, 16648, 16873, 17062, 17582, 17764, 18475, 19202, 19484, 21342, 21370, 21613, 23688, 23731, 24116, 24132, 25528, 26784, 26836, 27374, 27408, 28085, 28471, 30026, 30113, 30485, 30763, 31219, 31410, 32489, 32626, 33470, 34705, 35808, 36871, 37051, 37351, 37355, 38247, 39951, 40127, 40369, 41086, 41876, 41928, 43534, 43774, 43858, 43998, 44238, 44770, 45047, 46188, 47661, 48447, 48845, 49575, 51555, 51894, 51956, 52324, 52412, 52808, 53190, 54215, 55661, 56050, 57086, 58180, 58193, 58260, 58846, 59283, 60565, 60637, 60959, 61035, 61610, 61822, 62155, 62278, 62770, 62939, 64995, 65216, 66288, 67412, 67748, 67936, 67995, 68394, 68533, 68981, 69775, 69856, 70463, 71093, 71220, 72002, 72281, 72414, 72698, 74878, 76098, 79462, 79539, 79871, 80343, 80555, 80839, 80854, 81261, 81392, 82116, 82261, 8

In [7]:
print(len(cut_df_chunk[((1-cut_df_chunk["bdt_score_0"])>sq_BDT_cut)&((1-cut_df_chunk["bdt_score_1"])>sq_BDT_cut)]))

166


In [8]:
### Open saved ROOT file:
root_file = f"/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/{samples[0]}/chunk_2_with_bdtcut0-99965.root"
root_Rdf = ROOT.RDataFrame("events", root_file)
root_Rdf_np = root_Rdf.AsNumpy(columns= vars_to_save+["bdt_score_0","bdt_score_1","bdt_score_2"])
root_sub_df = pd.DataFrame(root_Rdf_np)


In [9]:
### Open saved ROOT file:
root_file_nocut = f"/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/{samples[0]}/chunk_2_with_evtid.root"
root_Rdf_nocut = ROOT.RDataFrame("events", root_file_nocut)
root_Rdf_np_nocut = root_Rdf_nocut.AsNumpy(columns= vars_to_save)
root_sub_df_nocut = pd.DataFrame(root_Rdf_np_nocut)


In [10]:
print(len(root_sub_df[((1-root_sub_df["bdt_score_0"])>sq_BDT_cut)&((1-root_sub_df["bdt_score_1"])>sq_BDT_cut)]))

23


In [11]:
print(list(root_sub_df[((1-root_sub_df["bdt_score_0"])>sq_BDT_cut)&((1-root_sub_df["bdt_score_1"])>sq_BDT_cut)]["evt_id"]))

[33859, 19159, 20224, 23149, 25213, 61784, 32156, 3228, 38337, 38367, 86, 1338, 2474, 44690, 56885, 66244, 68366, 70021, 73175, 73731, 75889, 87193, 90402]


In [12]:
root_sub_df[root_sub_df["evt_id"]==1338]

,MC_orivtx_y,MC_eta,Rec_true_M1ofM2,MCq1_pt,MC_D3,Rec_true_PDG,MCem_pz,MCfinal_phi,ln_Rec_track_absz0chi2_max_hemisEmin,EVT_hemisEmax_maxpChargedRP_p,...,MCq1_pz,EVT_hemisEmin_nNeutral,ln_Rec_track_absd0chi2_max_hemisEmin,Rec_true_eta,MC_M1,evt_id,chunk,bdt_score_0,bdt_score_1,bdt_score_2
11,"[-1.782526305760257e-05, -1.782526305760257e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 21, 21, 0, 21, 0, 0, 0, 0, 0, 0, 21, 0,...",8.58956,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-211.0, 11.0, -211.0, 211.0, 211.0, 211.0, 21...",45.593918,"[0.0, 0.0, 0.8553560376167297, 1.2617644071578...",5.204247,7.211031,...,44.519489,6.0,-0.233766,"[-2.3554861545562744, -1.6502604484558105, -2....","[-999, -999, 0, 1, 2, 0, 1, 4, 4, 8, 8, 7, 10,...",1338,2,0.0,0.0,0.0


In [13]:
cut_df_chunk[cut_df_chunk["evt_id"]==1338]

,MC_orivtx_y,MC_eta,Rec_true_M1ofM2,MCq1_pt,MC_D3,Rec_true_PDG,MCem_pz,MCfinal_phi,ln_Rec_track_absz0chi2_max_hemisEmin,EVT_hemisEmax_maxpChargedRP_p,...,EVT_hemisEmin_nNeutral,ln_Rec_track_absd0chi2_max_hemisEmin,Rec_true_eta,MC_M1,evt_id,chunk,decay,bdt_score_0,bdt_score_1,bdt_score_2
2115,"[-1.782526305760257e-05, -1.782526305760257e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 21, 21, 0, 21, 0, 0, 0, 0, 0, 0, 21, 0,...",8.58956,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-211.0, 11.0, -211.0, 211.0, 211.0, 211.0, 21...",45.593918,"[0.0, 0.0, 0.8553560376167297, 1.2617644071578...",5.204247,7.211031,...,6,-0.233766,"[-2.3554861545562744, -1.6502604484558105, -2....","[-999, -999, 0, 1, 2, 0, 1, 4, 4, 8, 8, 7, 10,...",1338,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000277,0.000319,0.999404


In [14]:
friend = f"/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/{samples[0]}/chunk_2_bdt_friend.root"
friend_Rdf = ROOT.RDataFrame("bdt_scores", friend)
friend_Rdf_np = friend_Rdf.AsNumpy(columns= ["bdt_score_0","bdt_score_1","bdt_score_2"]+ ["evt_id"] + ["chunk"])
friend_sub_df = pd.DataFrame(friend_Rdf_np)


In [15]:
friend_sub_df[friend_sub_df["evt_id"]==1338]

,bdt_score_0,bdt_score_1,bdt_score_2,evt_id,chunk
1338,0.000277,0.000319,0.999404,1338,2


In [16]:
friend_sub_df[friend_sub_df["evt_id"]==56820]

,bdt_score_0,bdt_score_1,bdt_score_2,evt_id,chunk
82001,0.000591,0.161218,0.838191,56820,2


In [17]:
cut_df_chunk[cut_df_chunk["evt_id"]==56820]

,MC_orivtx_y,MC_eta,Rec_true_M1ofM2,MCq1_pt,MC_D3,Rec_true_PDG,MCem_pz,MCfinal_phi,ln_Rec_track_absz0chi2_max_hemisEmin,EVT_hemisEmax_maxpChargedRP_p,...,EVT_hemisEmin_nNeutral,ln_Rec_track_absd0chi2_max_hemisEmin,Rec_true_eta,MC_M1,evt_id,chunk,decay,bdt_score_0,bdt_score_1,bdt_score_2
2592,"[-1.505862564954441e-05, -1.505862564954441e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 21, 0, 21, 0, 0, 21, 21, 0, 21, 0, 0, 0, 0...",13.880518,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-211.0, -211.0, 11.0, -211.0, 211.0, -211.0, ...",45.518082,"[-2.5756425857543945, 0.0, 0.0, -1.46578931808...",2.685863,10.144367,...,5,-1.933288,"[-1.8930754661560059, -0.9873091578483582, -1....","[-999, -999, 5, 6, 2, 0, 1, 4, 5, 0, 1, 7, 7, ...",56820,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.000045,0.000157,0.999798


In [18]:
for variable in bdtvars:
    print(root_sub_df_nocut[root_sub_df_nocut["evt_id"]==56820][variable])
    print(cut_df_chunk[cut_df_chunk["evt_id"]==56820][variable])

62834    0.95401
Name: EVT_unitThrust_z, dtype: float32
2592    0.95401
Name: EVT_unitThrust_z, dtype: float32
62834   -3.681525
Name: EVT_sum_Rec_px, dtype: float32
2592   -3.681525
Name: EVT_sum_Rec_px, dtype: float32
62834   -0.864213
Name: Rec_thrustCosTheta_ave_hemisEmax, dtype: float32
2592   -0.864213
Name: Rec_thrustCosTheta_ave_hemisEmax, dtype: float32
62834    10.0
Name: ln_Rec_vtx_d2PV_max_hemisEmin, dtype: float32
2592    10.0
Name: ln_Rec_vtx_d2PV_max_hemisEmin, dtype: float32
62834    4.950676
Name: ln_Rec_track_absz0chi2_max_hemisEmax, dtype: float32
2592    4.950676
Name: ln_Rec_track_absz0chi2_max_hemisEmax, dtype: float32
62834    0.772687
Name: Rec_thrustCosTheta_ave_hemisEmin, dtype: float32
2592    0.772687
Name: Rec_thrustCosTheta_ave_hemisEmin, dtype: float32
62834    0.714395
Name: Rec_PV_chi2, dtype: float32
2592    0.714395
Name: Rec_PV_chi2, dtype: float32
62834   -1.0
Name: EVT_hemisEmax_maxpChargedRP_fromPV_transformed, dtype: float32
2592   -1.0
Name: EVT

In [19]:
from df_makers.data_to_pickle_function import sanitize_filename, remove_dot_etc

## canibalising script which applies BDT to root files
runmode ="Bu2lnu_background_no_lepton_veto"
cut = None #'cut must be none to be able to reattach friend tree!!!!!S

#path to data and outputs
inputpath  = basic_functions.check_inputpath(cfg.fccana_opts["outputDir"][runmode])

if cut:
    cut_name = sanitize_filename(cut)
    outputpath   = basic_functions.set_outputpath(os.path.join(inputpath,f"root_bdtscores_cut{cut_name}")) 
else:
    outputpath   = basic_functions.set_outputpath(os.path.join(inputpath,"root_bdtscores"))

if BDT_params:
    #convert BDT cut value into a string(e.g., 0.999 -> "0999")
    #BDT_cut_str =str(BDT_cut_value).replace('.','')
    # Create the name
    #bdtcut_name = f"bdtlh_{BDT_cut_str}cut"
    bdtcut_name = "bdtlh_nocut"
    outputpath = basic_functions.set_outputpath(os.path.join(outputpath, bdtcut_name))


# print statements to check loading things expect
print(f"----> INFO: Loading files from")
print(f"{15*' '}{inputpath}")
print(f"----> INFO: Output will be saved to")
print(f"{15*' '}{outputpath}")

#calculating efficiencies and also saving files paths used to calculate efficiencies to ensure save same ones
selection_efficiency = efficiency_finder.get_efficiencies('custom',
                                                    further_analysis=True,
                                                    samples = samples,
                                                    raw=True, #ie. want full efficiency including tupling and prelim cuts
                                                    cut = cut,
                                                    custompath=inputpath,
                                                    verbose=False,
                                                    return_files_list=True)
#nb if want to specify a certain number of chunks to use do it in here and will follow through into filepaths

filepaths_dict = {key: value for key, value in selection_efficiency.items() if key.endswith('_files')}
efficiencies_dict = {key: value for key, value in selection_efficiency.items() if key.endswith('_eff')}
efficiencies_err_dict = {key: value for key, value in selection_efficiency.items() if key.endswith('_err')}


# going to print the efficiencies for each now so can manually check
print("Efficiencies:"+'\n')
eff_to_print = [[ key , efficiencies_dict[key+'_eff']] for key in samples]
print( tabulate(  eff_to_print, headers=["decay", "efficiency"] ) +'\n')


#########################################################################
## Collecting events into dfs and adding preselection efficiency columns
#########################################################################

#if BDT_params:
#    eff_bdtlh_cut = {}
#    N_before_bdtlh_cut = {}

# now collect relevant events into a dataframe
for decay in samples:
    print(f'Starting processing decay: {decay}')
    decay_outpath = basic_functions.set_outputpath(os.path.join(outputpath,f'{decay}'))
    eff = efficiencies_dict[decay+'_eff']
    filepaths = filepaths_dict[decay+'_files']
    populated_filepaths=[]

    #if BDT_params:
    #    N_pre=0
    #    N_post=0

    #only try and get events from files that have non-zero number of selected events
    for file in filepaths:
        # Read TParameter objects from the original ROOT file
        input_file = ROOT.TFile.Open(file)
        tparam = input_file.Get("eventsSelected")
        evt_selected_val = tparam.GetVal()
        if evt_selected_val  !=0:
            populated_filepaths.append(file)

    #do each file separately
    nchunks = len(populated_filepaths)

    for n in range(1):
        print(f'---> Starting processing chunk {n} of {nchunks}')
        file =populated_filepaths[n]
        filename = os.path.basename(file)
        print(filename)
        Rdf = ROOT.RDataFrame("events", file)

        if cut:
            Rdf = Rdf.Filter(cut)

        # Define a synthetic event ID
        Rdf = Rdf.Define("evt_id", "rdfentry_")

        # Also add a column with chunk number (use as fake run number!)
        # Extract chunk number from filename
        basename = os.path.basename(file)          
        chunk_str = basename.replace(".root", "")  
        chunk_num = int(chunk_str.split("_")[1])
        Rdf = Rdf.Define("chunk", str(chunk_num))

        # Snapshot with evt_id and chunk included
        out_file = os.path.join(decay_outpath, filename.replace(".root", "_with_evtid.root"))
        all_cols = list(Rdf.GetColumnNames())
        cols_to_save = [c for c in all_cols if c != "Rec_vtx_indRP"] #Rec_vtx_indRP problematic
        #Rdf.Snapshot("events", out_file, cols_to_save)  


        Rdf_np = Rdf.AsNumpy(columns= vars_to_save + ["evt_id"] + ["chunk"]) ##add vars that can be used as ID to vars to save so that later can create event ID and use to match events
        sub_df = pd.DataFrame(Rdf_np)
        sub_df["decay"] = decay
        sub_df["eff_presel"] = eff

        #print(Rdf_np["evt_id"])

        # want to make sure that integer types are actually set as integers - currenlty stored as float
        #if changed branches significantly might be worth checking the list is still right, with current branches expected integers in yaml
        integer_branches = [s for s in vars_to_save if '_n' in s and '_norm' not in s] + ["evt_id","chunk"]
        for integer_branch in integer_branches:
            sub_df[integer_branch] = sub_df[integer_branch].astype(np.int32) #must be int32 to match BDT training data

        #print(np.array(sub_df["evt_id"]))


        if BDT_params:

            #####################
            # apply BDT and cut #
            #####################
            
            model, bdtname, dataframe_chunk = load_bdt_and_apply(sub_df, 
                                config_bdtopts = BDT_params["config_bdtopts"],
                                training_round = BDT_params["training_round"],
                                hps_dict_name = BDT_params["hps_dict_name"],
                                features_list_name = BDT_params["features_list_name"],
                                bdt_label = BDT_params["bdt_label"])


dataframe_chunk

    


----> INFO: Loading files from
               /r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/
----> INFO: Output will be saved to
               /r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut


Efficiencies:

decay                                        efficiency
-----------------------------------------  ------------
p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu      0.925491

Starting processing decay: p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu
---> Starting processing chunk 0 of 10
chunk_2.root
Loading BDT model...
BDT model loaded successfully.


,MC_orivtx_y,MC_eta,Rec_true_M1ofM2,MCq1_pt,MC_D3,Rec_true_PDG,MCem_pz,MCfinal_phi,ln_Rec_track_absz0chi2_max_hemisEmin,EVT_hemisEmax_maxpChargedRP_p,...,ln_Rec_track_absd0chi2_max_hemisEmin,Rec_true_eta,MC_M1,evt_id,chunk,decay,eff_presel,bdt_score_0,bdt_score_1,bdt_score_2
0,"[-6.228215170267504e-06, -6.228215170267504e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 0, 0, 21, 0, 0, 0, 0, 0, 21, 0, 0, 0, 0...",42.151958,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-211.0, 211.0, 211.0, -211.0, -211.0, -13.0, ...",45.594002,"[0.0, 0.0, 0.13872528076171875, 0.032040949910...",5.509006,11.788661,...,5.223865,"[-0.41681015491485596, -0.25296539068222046, -...","[-999, -999, 0, 1, 2, 0, 1, 4, 4, 8, 8, 7, 10,...",0,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.925491,0.000610,0.791563,0.207827
1,"[-1.2146228982601315e-06, -1.2146228982601315e...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 0, 0, 0, 0, 0, 0, 21, 21, 0, 21, 0, 21,...",20.229643,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-211.0, -211.0, -211.0, 211.0, 211.0, -211.0,...",45.593754,"[0.0, 0.0, 2.726097822189331, -0.7930805683135...",2.431007,6.904299,...,2.569338,"[-1.6916476488113403, -0.05379219725728035, -1...","[-999, -999, 0, 1, 2, 0, 1, 4, 4, 7, 7, 8, 10,...",1,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.925491,0.004800,0.980797,0.014403
2,"[2.8096468668081798e-05, 2.8096468668081798e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 21, 0, 0, 21, 21, 21, 21, 21, 21, 0, 21, 0...",28.997864,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-13.0, -211.0, -211.0, 211.0, -321.0, -2212.0...",45.594002,"[0.0, 0.0, 1.8547688722610474, 0.6745105385780...",1.853182,6.451986,...,1.842324,"[-0.588418185710907, -0.0699211061000824, -0.0...","[-999, -999, 0, 1, 2, 0, 1, 4, 4, 8, 8, 7, 10,...",2,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.925491,0.011221,0.047458,0.941321
3,"[2.9628597985720262e-05, 2.9628597985720262e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",37.764572,"[-999, -999, -999, -999, -999, -999, -999, -99...","[211.0, -211.0, -211.0, 13.0, 2212.0, -211.0, ...",45.592297,"[-0.20498579740524292, 0.0, 0.0, -2.0415995121...",5.634098,13.085112,...,4.455166,"[-0.17793112993240356, -0.8515384793281555, -0...","[-999, -999, 5, 6, 2, 0, 1, 4, 5, 0, 1, 7, 7, ...",3,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.925491,0.000520,0.996808,0.002672
4,"[-2.9747450753347948e-05, -2.9747450753347948e...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 0, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 21...",42.831963,"[-999, -999, -999, -999, -999, -999, -999, -99...","[211.0, 211.0, 211.0, -211.0, -211.0, 211.0, -...",45.594002,"[0.8896371722221375, 0.0, 0.0, -2.835526943206...",4.506413,7.895433,...,3.882767,"[-0.577675461769104, -0.5140632390975952, -1.6...","[-999, -999, 5, 6, 2, 0, 1, 4, 6, 0, 1, 7, 7, ...",4,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.925491,0.000336,0.984643,0.015021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92501,"[5.0109680159948766e-05, 5.0109680159948766e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[21, 0, 21, 21, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",37.078655,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-321.0, -211.0, -321.0, 321.0, 321.0, 13.0, 2...",45.588867,"[-2.746249198913574, 0.0, 0.0, -1.161461830139...",3.347418,5.967246,...,3.218088,"[-0.6146131753921509, -1.1302615404129028, -0....","[-999, -999, 5, 6, 2, 0, 1, 4, 5, 0, 1, 7, 7, ...",92501,2,p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu,0.925491,0.006524,0.983044,0.010432
92502,"[-1.682758011156693e-05, -1.682758011156693e-0...","[99999997952.0, -99999997952.0, 99999997952.0,...","[0, 0, 0, 0, 21, 0, 21, 0, 0, 21, 21, 0, 21, 0...",35.203236,"[-999, -999, -999, -999, -999, -999, -999, -99...","[-211.0, -211.0, 211.0, 11.0, 2212.0, 211.0, -...",45.593727,"[0.0, 0.0, -0.3187055289745331, -

In [20]:
print(len(dataframe_chunk[((1-dataframe_chunk["bdt_score_0"])>sq_BDT_cut)&((1-dataframe_chunk["bdt_score_1"])>sq_BDT_cut)&(dataframe_chunk["EVT_hemisEmin_nLept"]==0)]))

166


In [22]:
print(file)

/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu/chunk_2.root


In [23]:
# NOw trying same thing but changing int32 to int64 (ie. what initially ran with..)


## canibalising script which applies BDT to root files
runmode ="Bu2lnu_background_no_lepton_veto"
cut = None #'cut must be none to be able to reattach friend tree!!!!!S

#path to data and outputs
inputpath  = basic_functions.check_inputpath(cfg.fccana_opts["outputDir"][runmode])


if cut:
    cut_name = sanitize_filename(cut)
    outputpath   = basic_functions.set_outputpath(os.path.join(inputpath,f"root_bdtscores_cut{cut_name}")) 
else:
    outputpath   = basic_functions.set_outputpath(os.path.join(inputpath,"root_bdtscores"))

if BDT_params:
    #convert BDT cut value into a string(e.g., 0.999 -> "0999")
    #BDT_cut_str =str(BDT_cut_value).replace('.','')
    # Create the name
    #bdtcut_name = f"bdtlh_{BDT_cut_str}cut"
    bdtcut_name = "bdtlh_nocut"
    outputpath = basic_functions.set_outputpath(os.path.join(outputpath, bdtcut_name))


# print statements to check loading things expect
print(f"----> INFO: Loading files from")
print(f"{15*' '}{inputpath}")
print(f"----> INFO: Output will be saved to")
print(f"{15*' '}{outputpath}")

#calculating efficiencies and also saving files paths used to calculate efficiencies to ensure save same ones
selection_efficiency = efficiency_finder.get_efficiencies('custom',
                                                    further_analysis=True,
                                                    samples = samples,
                                                    raw=True, #ie. want full efficiency including tupling and prelim cuts
                                                    cut = cut,
                                                    custompath=inputpath,
                                                    verbose=False,
                                                    return_files_list=True)
#nb if want to specify a certain number of chunks to use do it in here and will follow through into filepaths

filepaths_dict = {key: value for key, value in selection_efficiency.items() if key.endswith('_files')}
efficiencies_dict = {key: value for key, value in selection_efficiency.items() if key.endswith('_eff')}
efficiencies_err_dict = {key: value for key, value in selection_efficiency.items() if key.endswith('_err')}


# going to print the efficiencies for each now so can manually check
print("Efficiencies:"+'\n')
eff_to_print = [[ key , efficiencies_dict[key+'_eff']] for key in samples]
print( tabulate(  eff_to_print, headers=["decay", "efficiency"] ) +'\n')


#########################################################################
## Collecting events into dfs and adding preselection efficiency columns
#########################################################################

#if BDT_params:
#    eff_bdtlh_cut = {}
#    N_before_bdtlh_cut = {}

# now collect relevant events into a dataframe
for decay in samples:
    print(f'Starting processing decay: {decay}')
    decay_outpath = basic_functions.set_outputpath(os.path.join(outputpath,f'{decay}'))
    eff = efficiencies_dict[decay+'_eff']
    filepaths = filepaths_dict[decay+'_files']
    populated_filepaths=[]

    #if BDT_params:
    #    N_pre=0
    #    N_post=0

    #only try and get events from files that have non-zero number of selected events
    for file in filepaths:
        # Read TParameter objects from the original ROOT file
        input_file = ROOT.TFile.Open(file)
        tparam = input_file.Get("eventsSelected")
        evt_selected_val = tparam.GetVal()
        if evt_selected_val  !=0:
            populated_filepaths.append(file)

    #do each file separately
    nchunks = len(populated_filepaths)

    for n in range(1):
        print(f'---> Starting processing chunk {n} of {nchunks}')
        file =populated_filepaths[n]
        filename = os.path.basename(file)
        print(filename)
        Rdf = ROOT.RDataFrame("events", file)

        if cut:
            Rdf = Rdf.Filter(cut)

        # Define a synthetic event ID
        Rdf = Rdf.Define("evt_id", "rdfentry_")

        # Also add a column with chunk number (use as fake run number!)
        # Extract chunk number from filename
        basename = os.path.basename(file)          
        chunk_str = basename.replace(".root", "")  
        chunk_num = int(chunk_str.split("_")[1])
        Rdf = Rdf.Define("chunk", str(chunk_num))

        # Snapshot with evt_id and chunk included
        out_file = os.path.join(decay_outpath, filename.replace(".root", "_with_evtid.root"))
        all_cols = list(Rdf.GetColumnNames())
        cols_to_save = [c for c in all_cols if c != "Rec_vtx_indRP"] #Rec_vtx_indRP problematic and not needed for bdt
        Rdf.Snapshot("events", f"test_friend_{samples[0]}_chunk_2_with_evtid.root", cols_to_save)  


        Rdf_np = Rdf.AsNumpy(columns= vars_to_save + ["evt_id"] + ["chunk"]) ##add vars that can be used as ID to vars to save so that later can create event ID and use to match events
        sub_df = pd.DataFrame(Rdf_np)
        sub_df["decay"] = decay
        sub_df["eff_presel"] = eff

        #print(Rdf_np["evt_id"])

        # want to make sure that integer types are actually set as integers - currenlty stored as float
        #if changed branches significantly might be worth checking the list is still right, with current branches expected integers in yaml
        integer_branches = [s for s in vars_to_save if '_n' in s and '_norm' not in s] + ["evt_id","chunk"]
        for integer_branch in integer_branches:
            sub_df[integer_branch] = sub_df[integer_branch].astype(np.int64) #must be int32 to match BDT training data

        #print(np.array(sub_df["evt_id"]))


        if BDT_params:

            #####################
            # apply BDT and cut #
            #####################
            
            model, bdtname, dataframe_chunk = load_bdt_and_apply(sub_df, 
                                config_bdtopts = BDT_params["config_bdtopts"],
                                training_round = BDT_params["training_round"],
                                hps_dict_name = BDT_params["hps_dict_name"],
                                features_list_name = BDT_params["features_list_name"],
                                bdt_label = BDT_params["bdt_label"])
            
            #convert back to arrays
            bdt0 = dataframe_chunk['bdt_score_0'].to_numpy()
            bdt1 = dataframe_chunk['bdt_score_1'].to_numpy()
            bdt2 = dataframe_chunk['bdt_score_2'].to_numpy()
            chunk = dataframe_chunk['chunk'].to_numpy()
            evt_id = dataframe_chunk['evt_id'].to_numpy()

            ##############################################################################################
            # Save BDT scores into a friend tree + index branches
            ##############################################################################################
            # Create a new ROOT file for the friend tree
            friend_out = f"test_friend_{samples[0]}_chunk_2_bdt_friend.root"
            fout = ROOT.TFile(friend_out, "RECREATE")
            friend = ROOT.TTree("bdt_scores", "BDT output scores")

            # Buffers for branches
            bdt0_buf   = np.zeros(1, dtype=np.float64)
            bdt1_buf   = np.zeros(1, dtype=np.float64)
            bdt2_buf   = np.zeros(1, dtype=np.float64)
            evt_id_buf   = np.zeros(1, dtype=np.int32)
            chunk_buf   = np.zeros(1, dtype=np.int32)

            
            friend.Branch("bdt_score_0", bdt0_buf, "bdt_score_0/D")
            friend.Branch("bdt_score_1", bdt1_buf, "bdt_score_1/D")
            friend.Branch("bdt_score_2", bdt2_buf, "bdt_score_2/D")
            friend.Branch("evt_id", evt_id_buf, "evt_id/L")
            friend.Branch("chunk", chunk_buf, "chunk/L")

            # Fill the friend tree from your arrays
            for evtid, ch, v0, v1, v2 in zip(evt_id,chunk, bdt0, bdt1, bdt2):
                evt_id_buf[0] = evtid
                chunk_buf[0] = ch
                bdt0_buf[0]   = v0
                bdt1_buf[0]   = v1
                bdt2_buf[0]   = v2
                friend.Fill()

            friend.Write()
            fout.Close()


    
print(len((dataframe_chunk[((1-dataframe_chunk["bdt_score_0"])>sq_BDT_cut)&((1-dataframe_chunk["bdt_score_1"])>sq_BDT_cut)&(dataframe_chunk["EVT_hemisEmin_nLept"]==0)])))

----> INFO: Loading files from
               /r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/
----> INFO: Output will be saved to
               /r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut
Efficiencies:

decay                                        efficiency
-----------------------------------------  ------------
p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu      0.925491

Starting processing decay: p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu
---> Starting processing chunk 0 of 10
chunk_2.root
Loading BDT model...
BDT model loaded successfully.
166


In [24]:
bdtcut=0.99965
    

bdtcut_name = "bdtlh_nocut"    

trees = {}

for decay in samples:


    #get max number 
    numbers = []

    filename = f"test_friend_{samples[0]}_chunk_2.root"
    main_file = os.path.join(filename.replace(".root", "_with_evtid.root"))
    friend_file = os.path.join(filename.replace(".root", "_bdt_friend.root"))

    # Build the main chain
    main_chain = ROOT.TChain("events")
    main_chain.Add(main_file)

    # Build the friend chain
    friend_chain = ROOT.TChain("bdt_scores")
    friend_chain.Add(friend_file)

    # Build index so ROOT can match entries
    main_chain.BuildIndex("chunk","evt_id")
    friend_chain.BuildIndex("chunk","evt_id")
    

    # Attach friend chain to main chain
    main_chain.AddFriend(friend_chain)

    tree = main_chain

    n_events = tree.GetEntries()
    print(f"n events pre cuts: {n_events}")

    # Apply any pre-selection cuts that may not have been applied already
    cut_string = "EVT_hemisEmax_n > 10  && EVT_hemisEmin_nLept == 0"

    if bdtcut is not None:
        onemcut = 1-bdtcut
        bdtcut_string = f"bdt_score_1<{str(onemcut)} && bdt_score_0 <{str(onemcut)}"
        cut_string = bdtcut_string + "&&" + cut_string


    # Apply the cut: CopyTree returns a new TTree object
    cut_tree = main_chain.CopyTree(cut_string)

    print("Filtered tree saved with", cut_tree.GetEntries(), "entries")

cut_tree


n events pre cuts: 92506
Filtered tree saved with 166 entries


In [19]:
for variable in bdtvars:
    print(root_sub_df_nocut[root_sub_df_nocut["evt_id"]==56820][variable])
    print(cut_df_chunk[cut_df_chunk["evt_id"]==56820][variable])
    print(dataframe_chunk[dataframe_chunk["evt_id"]==56820][variable])

62834    31.663904
Name: EVT_Thrust_deltaE, dtype: float32
2592    31.663904
Name: EVT_Thrust_deltaE, dtype: float32
2592    31.663904
Name: EVT_Thrust_deltaE, dtype: float32
62834    3.318624
Name: EVT_hemisEmin_eNeutral, dtype: float32
2592    3.318624
Name: EVT_hemisEmin_eNeutral, dtype: float32
2592    3.318624
Name: EVT_hemisEmin_eNeutral, dtype: float32
62834    4.950676
Name: ln_Rec_track_absz0chi2_max_hemisEmax, dtype: float32
2592    4.950676
Name: ln_Rec_track_absz0chi2_max_hemisEmax, dtype: float32
2592    4.950676
Name: ln_Rec_track_absz0chi2_max_hemisEmax, dtype: float32
62834    3.576937
Name: EVT_hemisEmin_e, dtype: float32
2592    3.576937
Name: EVT_hemisEmin_e, dtype: float32
2592    3.576937
Name: EVT_hemisEmin_e, dtype: float32
62834    10.0
Name: ln_Rec_vtx_d2PV_min_hemisEmin, dtype: float32
2592    10.0
Name: ln_Rec_vtx_d2PV_min_hemisEmin, dtype: float32
2592    10.0
Name: ln_Rec_vtx_d2PV_min_hemisEmin, dtype: float32
62834    2.0
Name: Rec_vtx_n, dtype: float32
25